# 五个 Barra 风格因子的独立 Long-Short 收益

本 Notebook 分别构造 Market Beta、Size、Momentum、Volatility、Liquidity 五条 Long-Short 收益序列。每条序列内部的 Top 10% 与 Bottom 10% 股票分别等权，但五个风格因子之间绝不等权合成。

第一版工程默认：Beta 252日且 min_periods=120；Size 为 log(流通市值)；Momentum 为 252日回看并跳过最近21日；Volatility 为252日日收益波动率且 min_periods=120；Liquidity 为60日平均换手率。每个调仓日先对各风格暴露做1%/99%缩尾和 z-score，再独立构造多空组合。所有参数均可配置，不代表研报披露的内部定义。

In [ ]:
# 1. 环境、导入与集中配置
import sys
from dataclasses import asdict
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name.lower() == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from factor_gfn.barra import (
    BarraConfig, build_barra_auxiliary_arrays, build_barra_long_short_returns,
    cumulative_return, load_barra_factor_set, load_barra_inputs,
    run_barra_factor_pipeline, summarize_long_short,
)
from factor_gfn.evaluator.metrics import EvaluationConfig, build_forward_returns, select_rebalance_indices

barra_config = BarraConfig()
evaluation_config = EvaluationConfig()
display(pd.Series(asdict(barra_config), name='value').to_frame())

In [ ]:
# 2. 构造股本/市值辅助数组并计算五个风格暴露（长任务，只需成功运行一次）
# 确认日频预处理和历史股本下载完成后，将开关改为 True。不会修改 data/raw。
RUN_BARRA_BUILD = True

if RUN_BARRA_BUILD:
    auxiliary_summary = build_barra_auxiliary_arrays()
    display(auxiliary_summary)
    barra_metadata = run_barra_factor_pipeline(barra_config)
    display(barra_metadata)
else:
    print('未执行长任务；如尚未生成 data/processed/barra，请将 RUN_BARRA_BUILD 改为 True。')

In [ ]:
# 3. 加载结果并构造 open[t+6] / open[t+1] - 1 标签和统一5日调仓日期
inputs = load_barra_inputs()
factor_set = load_barra_factor_set()
forward_returns = build_forward_returns(inputs.open, evaluation_config)
base_count = np.sum(inputs.universe_mask & np.isfinite(forward_returns), axis=1)
rebalance_indices = select_rebalance_indices(base_count, evaluation_config)
print('矩阵形状:', forward_returns.shape)
print('调仓期数:', len(rebalance_indices))
print('首末调仓日:', inputs.dates[rebalance_indices[0]], inputs.dates[rebalance_indices[-1]])

In [ ]:
# 4. 五个因子各自独立的 Top 10% - Bottom 10% 收益与摘要
barra_ls = build_barra_long_short_returns(
    factor_set, forward_returns, rebalance_indices, barra_config
)
summary = pd.DataFrame({
    name: asdict(summarize_long_short(series.long_short_return))
    for name, series in barra_ls.items()
}).T
display(summary)

In [ ]:
# 5. 五条独立 Long-Short 累计收益曲线（不合成）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
fig, axes = plt.subplots(3, 2, figsize=(14, 13), sharex=True)
for ax, (name, series) in zip(axes.flat, barra_ls.items()):
    values = series.long_short_return
    valid = np.isfinite(values)
    curve = cumulative_return(values)
    ax.plot(inputs.dates[valid], curve[valid], label=name)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(name)
    ax.set_ylabel('累计 Long-Short 收益')
    ax.grid(alpha=0.25)
    ax.legend()
axes.flat[-1].axis('off')
fig.suptitle('五个 Barra 风格因子的独立5日 Long-Short 收益（不进行风格合成）', y=1.01)
plt.tight_layout()
plt.show()